In [1]:
import pandas as pd
import torch
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

# 메모리 초기화 함수
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

# 1. 데이터 로드
train_combined_path = '../Xuxeong/final_train_data2.csv'
df = pd.read_csv(train_combined_path)
df = df.dropna(subset=['conversation', 'label'])

# Train / Validation 분리 (8:2)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# 하이퍼파라미터 설정
MAX_LENGTH = 128
BATCH_SIZE = 32
MODEL_NAME = "beomi/KcELECTRA-base-v2022" 
num_labels = len(df['label'].unique()) 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dataset 클래스 정의
class KcElectraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = KcElectraDataset(train_df['conversation'].values, train_df['label'].values, tokenizer, MAX_LENGTH)
val_dataset = KcElectraDataset(val_df['conversation'].values, val_df['label'].values, tokenizer, MAX_LENGTH)

# 평가지표 함수
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

config.json:   0%|          | 0.00/504 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
clear_memory()
print("▶ [Seed 42] 학습 시작...")
model_42 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

args_42 = TrainingArguments(
    output_dir='./results_seed42',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=8e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

trainer_42 = Trainer(model=model_42, args=args_42, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
trainer_42.train()

trainer_42.save_model("./best_kcelectra_seed42")
tokenizer.save_pretrained("./best_kcelectra_seed42")
print("✅ Seed 42 저장 완료.")

▶ [Seed 42] 학습 시작...


pytorch_model.bin:   0%|          | 0.00/511M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Seed 42 저장 완료.


In [3]:
clear_memory()
print("▶ [Seed 2024] 학습 시작...")
model_2024 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

args_2024 = TrainingArguments(
    output_dir='./results_seed2024',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=8e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    seed=2024,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

trainer_2024 = Trainer(model=model_2024, args=args_2024, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
trainer_2024.train()

trainer_2024.save_model("./best_kcelectra_seed2024")
tokenizer.save_pretrained("./best_kcelectra_seed2024")
print("✅ Seed 2024 저장 완료.")

▶ [Seed 2024] 학습 시작...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.986892,0.796528,0.768986
2,No log,0.455333,0.899711,0.892710
3,No log,0.336014,0.910318,0.903049
4,0.722438,0.297975,0.918033,0.911623
5,0.722438,0.295197,0.913211,0.906082


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Seed 2024 저장 완료.


In [4]:
clear_memory()
print("▶ [Seed 777] 학습 시작...")
model_777 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

args_777 = TrainingArguments(
    output_dir='./results_seed777',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=8e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    seed=777,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

trainer_777 = Trainer(model=model_777, args=args_777, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
trainer_777.train()

trainer_777.save_model("./best_kcelectra_seed777")
tokenizer.save_pretrained("./best_kcelectra_seed777")
print("✅ Seed 777 저장 완료.")

▶ [Seed 777] 학습 시작...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.035995,0.631630,0.546720
2,No log,0.516067,0.895853,0.886297
3,No log,0.360980,0.905497,0.897463
4,0.760880,0.328595,0.909354,0.901838
5,0.760880,0.324793,0.910318,0.902910


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Seed 777 저장 완료.


In [7]:
import pandas as pd
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# 1. Test 데이터 로드 (경로를 맞춰주세요)
test_path = './data/test.csv'  # 실제 test.csv 경로
test_df = pd.read_csv(test_path)

MAX_LENGTH = 128
BATCH_SIZE = 32

# 학습해둔 3개의 모델 폴더 경로
model_paths = [
    "./best_kcelectra_seed42",   # 첫 번째 (seed 42)
    "./best_kcelectra_seed2024", # 두 번째
    "./best_kcelectra_seed777"   # 세 번째
]

# 2. Test용 Dataset 클래스 (라벨이 없으므로 text만 반환)
class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 3. 앙상블 확률을 누적할 배열 초기화 (데이터 수 x 클래스 수)
num_test_data = len(test_df)
num_classes = 5
ensemble_probs = np.zeros((num_test_data, num_classes))

# 4. 각 모델별로 예측 수행 후 확률 누적
for idx, path in enumerate(model_paths):
    print(f"\n▶ [{idx+1}/3] '{path}' 모델 추론 시작...")
    
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path).to(device)
    model.eval()
    
    test_dataset = TestDataset(test_df['conversation'].values, tokenizer, MAX_LENGTH)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model_probs = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Softmax를 통과시켜 0~1 사이의 확률값으로 변환
            probs = F.softmax(logits, dim=1).cpu().numpy()
            model_probs.extend(probs)
            
    # 누적 배열에 더하기
    ensemble_probs += np.array(model_probs)

# 5. 최종 예측 (3개 모델의 확률 합산 후 가장 높은 클래스 선택)
final_preds = np.argmax(ensemble_probs, axis=1)

# 6. 제출 파일 생성 (대회 양식에 맞게 컬럼명 수정: idx, target)
# 올려주신 양식을 보니 문자가 아닌 '숫자'로 제출하는 방식입니다.
test_df['target'] = final_preds

submission_path = './ensemble_submission.csv'
# test_df에 있는 'idx'와 새로 만든 'target' 컬럼만 추출하여 저장
test_df[['idx', 'target']].to_csv(submission_path, index=False) 

print(f"\n🎉 앙상블 완료! 최종 제출 파일이 '{submission_path}'로 저장되었습니다.")

Device: cuda

▶ [1/3] './best_kcelectra_seed42' 모델 추론 시작...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:03<00:00,  4.26it/s]



▶ [2/3] './best_kcelectra_seed2024' 모델 추론 시작...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:03<00:00,  4.17it/s]



▶ [3/3] './best_kcelectra_seed777' 모델 추론 시작...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:04<00:00,  3.99it/s]


🎉 앙상블 완료! 최종 제출 파일이 './ensemble_submission.csv'로 저장되었습니다.
